# M1 4분류 Label Taxonomy 재설계

## tl;dr

이번 노트북의 목적은 모델 성능 개선이 아니라, M1 기준 `normal / fault / task / activity` 4분류 학습이 가능한 label/window 구조를 확정하는 것이다.

- `efd_possible`은 최상위 class가 아니라 fault 내부 metadata로만 사용한다.
- `faults.csv`에 없는 fault disturbance는 `efd_possible=False`가 아니라 `efd_possible_unknown`으로 기록한다.
- normal label 35건은 회사 제공 기준을 유지한다.
- task/activity label 존재 여부, fault detail 매칭, window coverage, substation group CV 가능성을 함께 점검한다.


## Context & Methods

### Key Assumptions

- 모든 계산은 M1 metadata와 M1 operational CSV만 사용한다.
- 기존 `normal vs efd_possible/pre_event` 결과는 fault 내부 pre-event 보조 실험으로 남긴다.
- 이번 단계는 label taxonomy와 window 정책 설계이며, final model 저장이나 운영 배포는 하지 않는다.


In [1]:
from __future__ import annotations

from pathlib import Path
import warnings
import zipfile

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path.cwd()
DATA_DIR = ROOT / "05_데이터셋" / "PreDist"
ZIP_PATH = DATA_DIR / "predist_dataset.zip"
OUTPUT_DIR = ROOT / "07_데이터산출물"
NOTEBOOK_PATH = ROOT / "06_노트북" / "16_m1_event_taxonomy_4class_design.ipynb"

EVENT_TAXONOMY_AUDIT_PATH = OUTPUT_DIR / "m1_event_taxonomy_audit.csv"
LABEL_INDEX_PATH = OUTPUT_DIR / "m1_4class_label_candidate_index.csv"
WINDOW_AUDIT_PATH = OUTPUT_DIR / "m1_4class_window_policy_audit.csv"
FEASIBILITY_SUMMARY_PATH = OUTPUT_DIR / "m1_4class_dataset_feasibility_summary.csv"
BASELINE_METRICS_PATH = OUTPUT_DIR / "m1_4class_baseline_metrics.csv"
BASELINE_PREDICTIONS_PATH = OUTPUT_DIR / "m1_4class_baseline_predictions.csv"
REPORT_PATH = OUTPUT_DIR / "16_M1_event_taxonomy_4class_설계_보고서.md"

SOURCE_PREFIX = "manufacturer 1"
RANDOM_STATE = 42
COVERAGE_THRESHOLD = 0.95
MIN_CLASS_SAMPLES = 10
EXPECTED_ROWS_7D = 1008

EVENT20_ID = 20
EVENT34_ID = 34
EVENT67_ID = 67
EVENT69_ID = 69
HARD_NORMAL_TAG_BY_EVENT = {
    19: "review_required_normal",
    35: "hard_normal_metadata",
    48: "hard_normal_metadata",
    68: "review_required_normal",
}

ORIGINAL_SIGNALS = [
    "outdoor_temperature",
    "s_hc1_supply_temperature",
    "s_hc1_supply_temperature_setpoint",
    "p_hc1_return_temperature",
    "p_net_meter_energy",
    "p_net_meter_volume",
    "p_net_meter_heat_power",
    "p_net_meter_flow",
    "p_net_supply_temperature",
    "p_net_return_temperature",
]
DERIVED_SIGNALS = [
    "s_hc1_supply_temperature_error",
    "p_net_delta_temperature",
    "p_net_power_flow_ratio",
    "p_return_gap",
]
ALL_SIGNALS = ORIGINAL_SIGNALS + DERIVED_SIGNALS
SENSOR_STATS = ["mean", "std", "min", "max", "median", "last_minus_first", "missing_rate"]
TEMPORAL_STATS = [
    "last_1d_mean_minus_prev_6d_mean",
    "last_12h_mean_minus_prev_12h_mean",
    "last_6h_mean_minus_prev_6h_mean",
    "last_1d_std_minus_prev_6d_std",
]
BASE70_FEATURES = [f"{signal}__{stat}" for signal in ORIGINAL_SIGNALS for stat in SENSOR_STATS]
DERIVED_STAT_FEATURES = [f"{signal}__{stat}" for signal in DERIVED_SIGNALS for stat in SENSOR_STATS]
TEMPORAL_FEATURES = [f"{signal}__{stat}" for signal in ALL_SIGNALS for stat in TEMPORAL_STATS]
EXPANDED154_FEATURES = BASE70_FEATURES + DERIVED_STAT_FEATURES + TEMPORAL_FEATURES
CLASS_ORDER = ["normal", "fault", "task", "activity"]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert ZIP_PATH.exists(), ZIP_PATH
assert len(BASE70_FEATURES) == 70
assert len(EXPANDED154_FEATURES) == 154


## Data

In [2]:
def read_metadata(name: str) -> pd.DataFrame:
    with zipfile.ZipFile(ZIP_PATH) as zf:
        with zf.open(f"{SOURCE_PREFIX}/{name}") as file:
            return pd.read_csv(file, sep=";")


def parse_datetime_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    df = df.copy()
    for column in columns:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column], errors="coerce")
    return df


normal_events = parse_datetime_columns(
    read_metadata("normal_events.csv"),
    ["Event start", "Event end", "Training start", "Training end"],
)
disturbances = parse_datetime_columns(read_metadata("disturbances.csv"), ["Event start"])
faults = parse_datetime_columns(
    read_metadata("faults.csv"),
    ["Report date", "Possible anomaly start", "Possible anomaly end", "Training start", "Training end"],
)

normal_events["Event ID"] = normal_events["Event ID"].astype(int)
normal_events["substation ID"] = normal_events["substation ID"].astype(int)
disturbances["substation ID"] = disturbances["substation ID"].astype(int)
faults["Event ID"] = faults["Event ID"].astype(int)
faults["substation ID"] = faults["substation ID"].astype(int)

compact_summary = pd.read_csv(OUTPUT_DIR / "m1_compact_feature_set_summary.csv")
compact13_text = compact_summary.loc[
    compact_summary["feature_set"].eq("compact13_overlap"), "features"
].iloc[0]
COMPACT13_FEATURES = [feature for feature in compact13_text.split("|") if feature]
assert len(COMPACT13_FEATURES) == 13
assert set(COMPACT13_FEATURES).issubset(EXPANDED154_FEATURES)

print("normal events:", len(normal_events))
print("disturbances:", len(disturbances))
print(disturbances["type"].value_counts().to_string())
print("fault records:", len(faults))


normal events: 35
disturbances: 165
type
fault       67
activity    55
task        43
fault records: 33


In [3]:
fault_match_columns = [
    "Event ID",
    "substation ID",
    "Report date",
    "Possible anomaly start",
    "Possible anomaly end",
    "Training start",
    "Training end",
    "efd_possible",
    "Fault label",
    "Problem EN",
    "Monitoring potential",
]
fault_details = faults[fault_match_columns].rename(
    columns={
        "Event ID": "fault_event_id",
        "Report date": "fault_report_date",
        "Possible anomaly start": "possible_anomaly_start",
        "Possible anomaly end": "possible_anomaly_end",
        "Training start": "fault_training_start",
        "Training end": "fault_training_end",
        "Fault label": "fault_label",
    }
)

disturbance_index = disturbances.reset_index(drop=False).rename(columns={"index": "disturbance_row_id"})
disturbance_index["disturbance_row_id"] = disturbance_index["disturbance_row_id"].astype(int) + 1
disturbance_index = disturbance_index.rename(columns={"Event start": "event_start", "type": "event_type"})

disturbance_taxonomy = disturbance_index.merge(
    fault_details,
    left_on=["substation ID", "event_start"],
    right_on=["substation ID", "fault_report_date"],
    how="left",
)
disturbance_taxonomy["final_class"] = disturbance_taxonomy["event_type"]
disturbance_taxonomy["row_kind"] = "event"
disturbance_taxonomy["source_id"] = disturbance_taxonomy["disturbance_row_id"].map(lambda value: f"disturbance_{int(value):04d}")
disturbance_taxonomy["source_event_id"] = disturbance_taxonomy["fault_event_id"]
disturbance_taxonomy["fault_detail_available"] = disturbance_taxonomy["fault_event_id"].notna()
disturbance_taxonomy["fault_label_unknown"] = (
    disturbance_taxonomy["fault_detail_available"]
    & disturbance_taxonomy["fault_label"].fillna("").astype(str).str.strip().str.lower().eq("unknown")
)
disturbance_taxonomy["fault_label_known"] = (
    disturbance_taxonomy["fault_detail_available"]
    & disturbance_taxonomy["fault_label"].notna()
    & ~disturbance_taxonomy["fault_label_unknown"]
)
disturbance_taxonomy["efd_possible_true"] = disturbance_taxonomy["efd_possible"].eq(True)
disturbance_taxonomy["efd_possible_false"] = disturbance_taxonomy["efd_possible"].eq(False)
disturbance_taxonomy["efd_possible_unknown"] = disturbance_taxonomy["event_type"].eq("fault") & disturbance_taxonomy["efd_possible"].isna()
disturbance_taxonomy["pre_event_eligible"] = disturbance_taxonomy["efd_possible_true"]
disturbance_taxonomy["strict_pre_event_positive"] = (
    disturbance_taxonomy["efd_possible_true"]
    & disturbance_taxonomy["possible_anomaly_start"].notna()
    & disturbance_taxonomy["possible_anomaly_end"].notna()
    & disturbance_taxonomy["fault_training_start"].notna()
    & disturbance_taxonomy["fault_training_end"].notna()
)
disturbance_taxonomy["event20_low_coverage_flag"] = disturbance_taxonomy["fault_event_id"].eq(EVENT20_ID)
disturbance_taxonomy["event34_unknown_flag"] = disturbance_taxonomy["fault_event_id"].eq(EVENT34_ID)
disturbance_taxonomy["event69_unknown_training_end_missing_flag"] = disturbance_taxonomy["fault_event_id"].eq(EVENT69_ID)
disturbance_taxonomy["event67_long_anomaly_flag"] = disturbance_taxonomy["fault_event_id"].eq(EVENT67_ID)
disturbance_taxonomy["hard_normal_tag"] = ""

normal_taxonomy = normal_events.rename(
    columns={
        "Event ID": "source_event_id",
        "substation ID": "substation ID",
        "Event start": "event_start",
        "Event end": "event_end",
        "Training start": "normal_training_start",
        "Training end": "normal_training_end",
    }
).copy()
normal_taxonomy["row_kind"] = "normal"
normal_taxonomy["source_id"] = normal_taxonomy["source_event_id"].map(lambda value: f"normal_{int(value):04d}")
normal_taxonomy["disturbance_row_id"] = np.nan
normal_taxonomy["event_type"] = ""
normal_taxonomy["final_class"] = "normal"
normal_taxonomy["fault_event_id"] = np.nan
normal_taxonomy["fault_report_date"] = pd.NaT
normal_taxonomy["possible_anomaly_start"] = pd.NaT
normal_taxonomy["possible_anomaly_end"] = pd.NaT
normal_taxonomy["fault_training_start"] = pd.NaT
normal_taxonomy["fault_training_end"] = pd.NaT
normal_taxonomy["efd_possible"] = np.nan
normal_taxonomy["fault_label"] = ""
normal_taxonomy["Problem EN"] = ""
normal_taxonomy["Monitoring potential"] = np.nan
normal_taxonomy["fault_detail_available"] = False
normal_taxonomy["fault_label_unknown"] = False
normal_taxonomy["fault_label_known"] = False
normal_taxonomy["efd_possible_true"] = False
normal_taxonomy["efd_possible_false"] = False
normal_taxonomy["efd_possible_unknown"] = False
normal_taxonomy["pre_event_eligible"] = False
normal_taxonomy["strict_pre_event_positive"] = False
normal_taxonomy["event20_low_coverage_flag"] = False
normal_taxonomy["event34_unknown_flag"] = False
normal_taxonomy["event69_unknown_training_end_missing_flag"] = False
normal_taxonomy["event67_long_anomaly_flag"] = False
normal_taxonomy["hard_normal_tag"] = normal_taxonomy["source_event_id"].map(HARD_NORMAL_TAG_BY_EVENT).fillna("")

taxonomy_columns = [
    "row_kind",
    "source_id",
    "source_event_id",
    "disturbance_row_id",
    "substation ID",
    "event_start",
    "event_end",
    "event_type",
    "final_class",
    "fault_event_id",
    "fault_report_date",
    "fault_detail_available",
    "fault_label",
    "fault_label_known",
    "fault_label_unknown",
    "efd_possible",
    "efd_possible_true",
    "efd_possible_false",
    "efd_possible_unknown",
    "pre_event_eligible",
    "strict_pre_event_positive",
    "event20_low_coverage_flag",
    "event34_unknown_flag",
    "event69_unknown_training_end_missing_flag",
    "event67_long_anomaly_flag",
    "hard_normal_tag",
]
taxonomy_df = pd.concat(
    [
        normal_taxonomy.reindex(columns=taxonomy_columns),
        disturbance_taxonomy.reindex(columns=taxonomy_columns),
    ],
    ignore_index=True,
)
taxonomy_df = taxonomy_df.rename(columns={"substation ID": "substation_id"})
taxonomy_df.to_csv(EVENT_TAXONOMY_AUDIT_PATH, index=False, encoding="utf-8-sig")

print("taxonomy rows:", len(taxonomy_df))
print(taxonomy_df["final_class"].value_counts().reindex(CLASS_ORDER).to_string())
print("fault detail available:", int(taxonomy_df["fault_detail_available"].sum()))


taxonomy rows: 200
final_class
normal      35
fault       67
task        43
activity    55
fault detail available: 33


In [4]:
OPERATIONAL_CACHE: dict[int, pd.DataFrame] = {}


def load_operational(substation_id: int) -> pd.DataFrame:
    path = f"{SOURCE_PREFIX}/operational_data/substation_{int(substation_id)}.csv"
    with zipfile.ZipFile(ZIP_PATH) as zf:
        with zf.open(path) as file:
            df = pd.read_csv(file, sep=";")
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    missing_columns = [column for column in ORIGINAL_SIGNALS if column not in df.columns]
    if missing_columns:
        raise ValueError(f"missing signal columns for substation {substation_id}: {missing_columns}")
    return df.sort_values("timestamp").reset_index(drop=True)


def get_operational(substation_id: int) -> pd.DataFrame:
    substation_id = int(substation_id)
    if substation_id not in OPERATIONAL_CACHE:
        OPERATIONAL_CACHE[substation_id] = load_operational(substation_id)
    return OPERATIONAL_CACHE[substation_id]


def add_derived_signals(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["s_hc1_supply_temperature_error"] = (
        pd.to_numeric(df["s_hc1_supply_temperature"], errors="coerce")
        - pd.to_numeric(df["s_hc1_supply_temperature_setpoint"], errors="coerce")
    )
    df["p_net_delta_temperature"] = (
        pd.to_numeric(df["p_net_supply_temperature"], errors="coerce")
        - pd.to_numeric(df["p_net_return_temperature"], errors="coerce")
    )
    flow = pd.to_numeric(df["p_net_meter_flow"], errors="coerce")
    heat_power = pd.to_numeric(df["p_net_meter_heat_power"], errors="coerce")
    df["p_net_power_flow_ratio"] = np.where(flow.abs() > 1e-9, heat_power / flow, np.nan)
    df["p_return_gap"] = (
        pd.to_numeric(df["p_hc1_return_temperature"], errors="coerce")
        - pd.to_numeric(df["p_net_return_temperature"], errors="coerce")
    )
    return df


def window_slice(substation_id: int, window_start: pd.Timestamp, window_end: pd.Timestamp) -> pd.DataFrame:
    df = get_operational(substation_id)
    mask = df["timestamp"].ge(window_start) & df["timestamp"].lt(window_end)
    return add_derived_signals(df.loc[mask, ["timestamp", *ORIGINAL_SIGNALS]].copy())


def expected_rows(window_start: pd.Timestamp, window_end: pd.Timestamp) -> int:
    minutes = (window_end - window_start).total_seconds() / 60
    return int(round(minutes / 10))


def event_overlap_summary(
    substation_id: int,
    window_start: pd.Timestamp,
    window_end: pd.Timestamp,
    self_disturbance_row_id,
) -> tuple[int, str]:
    events = disturbance_index.loc[
        disturbance_index["substation ID"].eq(int(substation_id))
        & disturbance_index["event_start"].ge(window_start)
        & disturbance_index["event_start"].lt(window_end)
    ].copy()
    if pd.notna(self_disturbance_row_id):
        events = events.loc[~events["disturbance_row_id"].eq(int(self_disturbance_row_id))]
    if events.empty:
        return 0, ""
    return int(len(events)), "|".join(sorted(events["event_type"].dropna().astype(str).unique()))


def safe_std(series: pd.Series) -> float:
    non_null = pd.to_numeric(series, errors="coerce").dropna()
    if len(non_null) > 1:
        return float(non_null.std(ddof=1))
    if len(non_null) == 1:
        return 0.0
    return np.nan


def stat_values(window_data: pd.DataFrame, signals: list[str]) -> dict[str, float]:
    values: dict[str, float] = {}
    for signal in signals:
        series = pd.to_numeric(window_data[signal], errors="coerce")
        non_null = series.dropna()
        values[f"{signal}__mean"] = float(non_null.mean()) if len(non_null) else np.nan
        values[f"{signal}__std"] = safe_std(series)
        values[f"{signal}__min"] = float(non_null.min()) if len(non_null) else np.nan
        values[f"{signal}__max"] = float(non_null.max()) if len(non_null) else np.nan
        values[f"{signal}__median"] = float(non_null.median()) if len(non_null) else np.nan
        if len(non_null) > 1:
            values[f"{signal}__last_minus_first"] = float(non_null.iloc[-1] - non_null.iloc[0])
        elif len(non_null) == 1:
            values[f"{signal}__last_minus_first"] = 0.0
        else:
            values[f"{signal}__last_minus_first"] = np.nan
        values[f"{signal}__missing_rate"] = float(series.isna().mean()) if len(series) else 1.0
    return values


def segment_mean(window_data: pd.DataFrame, signal: str, start: pd.Timestamp, end: pd.Timestamp) -> float:
    segment = window_data.loc[
        window_data["timestamp"].ge(start) & window_data["timestamp"].lt(end),
        signal,
    ]
    non_null = pd.to_numeric(segment, errors="coerce").dropna()
    return float(non_null.mean()) if len(non_null) else np.nan


def segment_std(window_data: pd.DataFrame, signal: str, start: pd.Timestamp, end: pd.Timestamp) -> float:
    segment = window_data.loc[
        window_data["timestamp"].ge(start) & window_data["timestamp"].lt(end),
        signal,
    ]
    return safe_std(segment)


def temporal_values(window_data: pd.DataFrame, window_start: pd.Timestamp, window_end: pd.Timestamp) -> dict[str, float]:
    values: dict[str, float] = {}
    cut_1d = window_end - pd.Timedelta(days=1)
    cut_12h = window_end - pd.Timedelta(hours=12)
    cut_24h = window_end - pd.Timedelta(hours=24)
    cut_6h = window_end - pd.Timedelta(hours=6)
    cut_12h_for_6h = window_end - pd.Timedelta(hours=12)
    for signal in ALL_SIGNALS:
        values[f"{signal}__last_1d_mean_minus_prev_6d_mean"] = (
            segment_mean(window_data, signal, cut_1d, window_end)
            - segment_mean(window_data, signal, window_start, cut_1d)
        )
        values[f"{signal}__last_12h_mean_minus_prev_12h_mean"] = (
            segment_mean(window_data, signal, cut_12h, window_end)
            - segment_mean(window_data, signal, cut_24h, cut_12h)
        )
        values[f"{signal}__last_6h_mean_minus_prev_6h_mean"] = (
            segment_mean(window_data, signal, cut_6h, window_end)
            - segment_mean(window_data, signal, cut_12h_for_6h, cut_6h)
        )
        values[f"{signal}__last_1d_std_minus_prev_6d_std"] = (
            segment_std(window_data, signal, cut_1d, window_end)
            - segment_std(window_data, signal, window_start, cut_1d)
        )
    return values


In [5]:
def candidate_window_specs(row: pd.Series) -> list[dict]:
    event_time = pd.to_datetime(row["event_start"])
    specs = []
    if row["final_class"] == "normal":
        specs.append(
            {
                "window_policy": "normal_event_7d",
                "window_start": pd.to_datetime(row["event_start"]),
                "window_end": pd.to_datetime(row["event_end"]),
                "selected_for_baseline": True,
            }
        )
    elif row["final_class"] == "fault":
        specs.append(
            {
                "window_policy": "fault_pre_7d",
                "window_start": event_time - pd.Timedelta(days=7),
                "window_end": event_time,
                "selected_for_baseline": True,
            }
        )
    elif row["final_class"] in {"task", "activity"}:
        specs.extend(
            [
                {
                    "window_policy": f"{row['final_class']}_pre_7d",
                    "window_start": event_time - pd.Timedelta(days=7),
                    "window_end": event_time,
                    "selected_for_baseline": True,
                },
                {
                    "window_policy": f"{row['final_class']}_post_7d",
                    "window_start": event_time,
                    "window_end": event_time + pd.Timedelta(days=7),
                    "selected_for_baseline": False,
                },
            ]
        )
    return specs


def make_window_row(row: pd.Series, spec: dict) -> dict:
    window_start = spec["window_start"]
    window_end = spec["window_end"]
    substation_id = int(row["substation_id"])
    window_data = window_slice(substation_id, window_start, window_end)
    expected = expected_rows(window_start, window_end)
    sample_count = int(len(window_data))
    coverage_rate = float(sample_count / expected) if expected else 0.0
    overlap_count, overlap_types = event_overlap_summary(
        substation_id,
        window_start,
        window_end,
        row["disturbance_row_id"],
    )
    base = {
        "source_id": row["source_id"],
        "row_kind": row["row_kind"],
        "final_class": row["final_class"],
        "event_type": row["event_type"],
        "source_event_id": row["source_event_id"],
        "disturbance_row_id": row["disturbance_row_id"],
        "substation_id": substation_id,
        "event_start": row["event_start"],
        "window_policy": spec["window_policy"],
        "window_start": window_start,
        "window_end": window_end,
        "window_days": float((window_end - window_start).total_seconds() / 86400),
        "sample_count": sample_count,
        "expected_sample_count": expected,
        "coverage_rate": coverage_rate,
        "coverage_eligible": bool(coverage_rate >= COVERAGE_THRESHOLD),
        "overlap_disturbance_count": overlap_count,
        "overlap_disturbance_types": overlap_types,
        "selected_for_baseline": bool(spec["selected_for_baseline"]),
        "baseline_included": bool(spec["selected_for_baseline"] and coverage_rate >= COVERAGE_THRESHOLD),
        "fault_event_id": row["fault_event_id"],
        "fault_detail_available": bool(row["fault_detail_available"]),
        "fault_label": row["fault_label"],
        "fault_label_known": bool(row["fault_label_known"]),
        "fault_label_unknown": bool(row["fault_label_unknown"]),
        "efd_possible": row["efd_possible"],
        "efd_possible_true": bool(row["efd_possible_true"]),
        "efd_possible_false": bool(row["efd_possible_false"]),
        "efd_possible_unknown": bool(row["efd_possible_unknown"]),
        "pre_event_eligible": bool(row["pre_event_eligible"]),
        "strict_pre_event_positive": bool(row["strict_pre_event_positive"]),
        "event20_low_coverage_flag": bool(row["event20_low_coverage_flag"]),
        "event34_unknown_flag": bool(row["event34_unknown_flag"]),
        "event69_unknown_training_end_missing_flag": bool(row["event69_unknown_training_end_missing_flag"]),
        "event67_long_anomaly_flag": bool(row["event67_long_anomaly_flag"]),
        "hard_normal_tag": row["hard_normal_tag"],
    }
    base.update(stat_values(window_data, ORIGINAL_SIGNALS))
    base.update(stat_values(window_data, DERIVED_SIGNALS))
    base.update(temporal_values(window_data, window_start, window_end))
    return base


window_rows = []
for row in taxonomy_df.itertuples(index=False):
    row_series = pd.Series(row._asdict())
    for spec in candidate_window_specs(row_series):
        window_rows.append(make_window_row(row_series, spec))

window_audit_df = pd.DataFrame(window_rows)
window_audit_df.to_csv(WINDOW_AUDIT_PATH, index=False, encoding="utf-8-sig")

label_index_df = window_audit_df.loc[window_audit_df["selected_for_baseline"]].copy()
label_index_df = label_index_df.sort_values(["final_class", "source_id"]).reset_index(drop=True)
label_index_df.to_csv(LABEL_INDEX_PATH, index=False, encoding="utf-8-sig")

print("window audit rows:", len(window_audit_df))
print("label index rows:", len(label_index_df))
print(label_index_df.groupby("final_class").agg(rows=("source_id", "size"), coverage_eligible=("coverage_eligible", "sum"), baseline_included=("baseline_included", "sum")).to_string())


window audit rows: 298
label index rows: 200
             rows  coverage_eligible  baseline_included
final_class                                            
activity       55                 47                 47
fault          67                 62                 62
normal         35                 35                 35
task           43                 37                 37


## Results

In [6]:
def make_model(model_name: str) -> Pipeline:
    if model_name == "dummy_most_frequent":
        return Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("model", DummyClassifier(strategy="most_frequent")),
            ]
        )
    if model_name == "logistic_balanced":
        return Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        class_weight="balanced",
                        max_iter=4000,
                        solver="lbfgs",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        )
    raise ValueError(model_name)


def evaluate_feature_set(feature_set: str, feature_columns: list[str]):
    data = label_index_df.loc[label_index_df["baseline_included"]].copy().reset_index(drop=True)
    X = data[feature_columns]
    y = data["final_class"].astype(str).to_numpy()
    groups = data["substation_id"].astype(int).to_numpy()
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    metric_rows = []
    prediction_rows = []
    for fold, (train_idx, test_idx) in enumerate(splitter.split(X, y, groups), start=1):
        train_groups = set(groups[train_idx])
        test_groups = set(groups[test_idx])
        assert train_groups.isdisjoint(test_groups)
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        for model_name in ["dummy_most_frequent", "logistic_balanced"]:
            model = make_model(model_name)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            report = classification_report(
                y_test,
                y_pred,
                labels=CLASS_ORDER,
                output_dict=True,
                zero_division=0,
            )
            overall = {
                "feature_set": feature_set,
                "feature_count": len(feature_columns),
                "model": model_name,
                "fold": fold,
                "metric_scope": "overall",
                "class_label": "all",
                "rows": int(len(y_test)),
                "accuracy": float(accuracy_score(y_test, y_pred)),
                "balanced_accuracy": float(balanced_accuracy_score(y_test, y_pred)),
                "macro_f1": float(f1_score(y_test, y_pred, labels=CLASS_ORDER, average="macro", zero_division=0)),
                "precision": float(report["macro avg"]["precision"]),
                "recall": float(report["macro avg"]["recall"]),
                "f1": float(report["macro avg"]["f1-score"]),
                "support": int(report["macro avg"]["support"]),
                "train_rows": int(len(train_idx)),
                "test_rows": int(len(test_idx)),
                "train_groups": "|".join(map(str, sorted(train_groups))),
                "test_groups": "|".join(map(str, sorted(test_groups))),
                "group_overlap_count": 0,
            }
            metric_rows.append(overall)
            for class_label in CLASS_ORDER:
                class_report = report.get(class_label, {"precision": 0, "recall": 0, "f1-score": 0, "support": 0})
                metric_rows.append(
                    {
                        **overall,
                        "metric_scope": "class",
                        "class_label": class_label,
                        "accuracy": np.nan,
                        "balanced_accuracy": np.nan,
                        "macro_f1": np.nan,
                        "precision": float(class_report["precision"]),
                        "recall": float(class_report["recall"]),
                        "f1": float(class_report["f1-score"]),
                        "support": int(class_report["support"]),
                    }
                )
            test_meta = data.iloc[test_idx].copy()
            for meta_row, pred_label in zip(test_meta.itertuples(index=False), y_pred):
                prediction_rows.append(
                    {
                        "feature_set": feature_set,
                        "feature_count": len(feature_columns),
                        "model": model_name,
                        "fold": fold,
                        "source_id": meta_row.source_id,
                        "final_class": meta_row.final_class,
                        "predicted_class": pred_label,
                        "correct": bool(meta_row.final_class == pred_label),
                        "substation_id": int(meta_row.substation_id),
                        "source_event_id": meta_row.source_event_id,
                        "disturbance_row_id": meta_row.disturbance_row_id,
                        "window_policy": meta_row.window_policy,
                        "coverage_rate": float(meta_row.coverage_rate),
                        "overlap_disturbance_count": int(meta_row.overlap_disturbance_count),
                        "fault_event_id": meta_row.fault_event_id,
                        "fault_detail_available": bool(meta_row.fault_detail_available),
                        "efd_possible_true": bool(meta_row.efd_possible_true),
                        "efd_possible_false": bool(meta_row.efd_possible_false),
                        "efd_possible_unknown": bool(meta_row.efd_possible_unknown),
                        "hard_normal_tag": meta_row.hard_normal_tag,
                    }
                )
    return metric_rows, prediction_rows


all_metric_rows = []
all_prediction_rows = []
for feature_set, feature_columns in {
    "compact13": COMPACT13_FEATURES,
    "expanded154": EXPANDED154_FEATURES,
}.items():
    metric_part, prediction_part = evaluate_feature_set(feature_set, feature_columns)
    all_metric_rows.extend(metric_part)
    all_prediction_rows.extend(prediction_part)

baseline_metrics_df = pd.DataFrame(all_metric_rows)
baseline_predictions_df = pd.DataFrame(all_prediction_rows)
baseline_metrics_df.to_csv(BASELINE_METRICS_PATH, index=False, encoding="utf-8-sig")
baseline_predictions_df.to_csv(BASELINE_PREDICTIONS_PATH, index=False, encoding="utf-8-sig")

baseline_metrics_df.loc[
    baseline_metrics_df["metric_scope"].eq("overall")
].groupby(["feature_set", "model"]).agg(
    balanced_accuracy=("balanced_accuracy", "mean"),
    macro_f1=("macro_f1", "mean"),
).reset_index()


,feature_set,model,balanced_accuracy,macro_f1
0,compact13,dummy_most_frequent,0.250000,0.127619
1,compact13,logistic_balanced,0.446562,0.435648
2,expanded154,dummy_most_frequent,0.250000,0.127619
3,expanded154,logistic_balanced,0.499150,0.471142


In [7]:
source_counts = taxonomy_df["final_class"].value_counts().reindex(CLASS_ORDER, fill_value=0)
selected_counts = label_index_df.groupby("final_class").agg(
    selected_rows=("source_id", "size"),
    coverage_eligible_rows=("coverage_eligible", "sum"),
    baseline_included_rows=("baseline_included", "sum"),
    substations=("substation_id", "nunique"),
    overlap_rows=("overlap_disturbance_count", lambda values: int((values > 0).sum())),
).reindex(CLASS_ORDER)

overall_metrics = baseline_metrics_df.loc[baseline_metrics_df["metric_scope"].eq("overall")].groupby(
    ["feature_set", "model"], as_index=False
).agg(
    balanced_accuracy=("balanced_accuracy", "mean"),
    macro_f1=("macro_f1", "mean"),
    accuracy=("accuracy", "mean"),
)
compact_logistic = overall_metrics.loc[
    overall_metrics["feature_set"].eq("compact13")
    & overall_metrics["model"].eq("logistic_balanced")
].iloc[0]
expanded_logistic = overall_metrics.loc[
    overall_metrics["feature_set"].eq("expanded154")
    & overall_metrics["model"].eq("logistic_balanced")
].iloc[0]
dummy_compact = overall_metrics.loc[
    overall_metrics["feature_set"].eq("compact13")
    & overall_metrics["model"].eq("dummy_most_frequent")
].iloc[0]

has_all_classes = bool((source_counts[CLASS_ORDER] > 0).all())
class_sample_sufficient = bool((selected_counts["baseline_included_rows"].fillna(0) >= MIN_CLASS_SAMPLES).all())
compact_lift_over_dummy = float(compact_logistic["macro_f1"] - dummy_compact["macro_f1"])
expanded_lift_over_compact = float(expanded_logistic["macro_f1"] - compact_logistic["macro_f1"])
task_exists = int(source_counts["task"]) > 0
activity_exists = int(source_counts["activity"]) > 0

if has_all_classes and class_sample_sufficient and compact_lift_over_dummy >= 0.10:
    final_decision = "M1_4class_possible"
elif has_all_classes and class_sample_sufficient and expanded_lift_over_compact >= 0.05:
    final_decision = "M1_4class_possible_but_feature_expansion_required"
elif int(source_counts["task"]) == 0 or int(selected_counts.loc["task", "baseline_included_rows"]) < MIN_CLASS_SAMPLES:
    final_decision = "M1_3class_possible"
else:
    final_decision = "M1_binary_or_3class_realistic"

summary_rows = []
summary_rows.append(
    {
        "section": "source_count",
        "item": "normal_events",
        "value": int(len(normal_events)),
        "note": "company provided normal label",
        "final_decision": final_decision,
    }
)
for class_label in CLASS_ORDER:
    summary_rows.append(
        {
            "section": "source_count",
            "item": class_label,
            "value": int(source_counts[class_label]),
            "note": "taxonomy source rows",
            "final_decision": final_decision,
        }
    )
summary_rows.extend(
    [
        {
            "section": "fault_detail",
            "item": "fault_disturbance_rows",
            "value": int(source_counts["fault"]),
            "note": "disturbances type=fault",
            "final_decision": final_decision,
        },
        {
            "section": "fault_detail",
            "item": "faults_csv_matched",
            "value": int(taxonomy_df["fault_detail_available"].sum()),
            "note": "exact match on substation and event time",
            "final_decision": final_decision,
        },
        {
            "section": "fault_detail",
            "item": "fault_detail_missing",
            "value": int(taxonomy_df["efd_possible_unknown"].sum()),
            "note": "recorded as efd_possible_unknown, not False",
            "final_decision": final_decision,
        },
        {
            "section": "fault_detail",
            "item": "fault_label_known",
            "value": int(taxonomy_df["fault_label_known"].sum()),
            "note": "fault metadata only",
            "final_decision": final_decision,
        },
        {
            "section": "fault_detail",
            "item": "fault_label_unknown",
            "value": int(taxonomy_df["fault_label_unknown"].sum()),
            "note": "fault metadata only",
            "final_decision": final_decision,
        },
        {
            "section": "fault_detail",
            "item": "efd_possible_true",
            "value": int(taxonomy_df["efd_possible_true"].sum()),
            "note": "fault metadata only",
            "final_decision": final_decision,
        },
        {
            "section": "fault_detail",
            "item": "efd_possible_false",
            "value": int(taxonomy_df["efd_possible_false"].sum()),
            "note": "fault metadata only",
            "final_decision": final_decision,
        },
        {
            "section": "baseline",
            "item": "compact13_macro_f1",
            "value": float(compact_logistic["macro_f1"]),
            "note": "group CV logistic sanity check",
            "final_decision": final_decision,
        },
        {
            "section": "baseline",
            "item": "expanded154_macro_f1",
            "value": float(expanded_logistic["macro_f1"]),
            "note": "comparison only",
            "final_decision": final_decision,
        },
        {
            "section": "baseline",
            "item": "compact13_lift_over_dummy",
            "value": compact_lift_over_dummy,
            "note": "macro f1 lift",
            "final_decision": final_decision,
        },
    ]
)
for class_label, row in selected_counts.iterrows():
    summary_rows.append(
        {
            "section": "window_policy",
            "item": f"{class_label}_baseline_included",
            "value": int(row["baseline_included_rows"]),
            "note": f"coverage >= {COVERAGE_THRESHOLD}",
            "final_decision": final_decision,
        }
    )
    summary_rows.append(
        {
            "section": "window_policy",
            "item": f"{class_label}_overlap_rows",
            "value": int(row["overlap_rows"]),
            "note": "rows with another disturbance in selected window",
            "final_decision": final_decision,
        }
    )

feasibility_summary_df = pd.DataFrame(summary_rows)
feasibility_summary_df.to_csv(FEASIBILITY_SUMMARY_PATH, index=False, encoding="utf-8-sig")
feasibility_summary_df


,section,item,value,note,final_decision
0,source_count,normal_events,35.000000,company provided normal label,M1_4class_possible
1,source_count,normal,35.000000,taxonomy source rows,M1_4class_possible
2,source_count,fault,67.000000,taxonomy source rows,M1_4class_possible
3,source_count,task,43.000000,taxonomy source rows,M1_4class_possible
4,source_count,activity,55.000000,taxonomy source rows,M1_4class_possible
5,fault_detail,fault_disturbance_rows,67.000000,disturbances type=fault,M1_4class_possible
6,fault_detail,faults_csv_matched,33.000000,exact match on substation and event time,M1_4class_possible
7,fault_detail,fault_detail_missing,34.000000,"recorded as efd_possible_unknown, not False",M1_4class_possible
8,fault_detail,fault_label_known,31.000000,fault metadata only,M1_4class_possible
9,fault_detail,fault_label_unknown,2.000000,fault metadata only,M1_4class_possible


## Takeaways

In [8]:
def markdown_table(df: pd.DataFrame, columns: list[str]) -> str:
    table = df[columns].copy()
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = []
    for _, row in table.iterrows():
        rows.append("| " + " | ".join(str(row[col]) for col in columns) + " |")
    return "\n".join([header, sep] + rows)


source_table = pd.DataFrame(
    {
        "class": CLASS_ORDER,
        "source_rows": [int(source_counts[label]) for label in CLASS_ORDER],
        "baseline_included": [int(selected_counts.loc[label, "baseline_included_rows"]) for label in CLASS_ORDER],
        "substations": [int(selected_counts.loc[label, "substations"]) for label in CLASS_ORDER],
        "overlap_rows": [int(selected_counts.loc[label, "overlap_rows"]) for label in CLASS_ORDER],
    }
)

fault_table = pd.DataFrame(
    [
        ["fault disturbance", int(source_counts["fault"])],
        ["faults.csv matched", int(taxonomy_df["fault_detail_available"].sum())],
        ["fault detail missing", int(taxonomy_df["efd_possible_unknown"].sum())],
        ["fault label known", int(taxonomy_df["fault_label_known"].sum())],
        ["fault label unknown", int(taxonomy_df["fault_label_unknown"].sum())],
        ["efd_possible True", int(taxonomy_df["efd_possible_true"].sum())],
        ["efd_possible False", int(taxonomy_df["efd_possible_false"].sum())],
    ],
    columns=["item", "count"],
)

metric_table = overall_metrics.copy()
for col in ["balanced_accuracy", "macro_f1", "accuracy"]:
    metric_table[col] = metric_table[col].map(lambda value: f"{float(value):.4f}")

window_policy_table = window_audit_df.groupby(["final_class", "window_policy"], as_index=False).agg(
    rows=("source_id", "size"),
    coverage_eligible=("coverage_eligible", "sum"),
    overlap_rows=("overlap_disturbance_count", lambda values: int((values > 0).sum())),
    median_coverage=("coverage_rate", "median"),
)
window_policy_table["median_coverage"] = window_policy_table["median_coverage"].map(lambda value: f"{float(value):.4f}")

report = f"""# M1 4분류 Label Taxonomy 설계 보고서

## 개요

이번 단계는 모델 성능 개선이 아니라, M1 기준 `normal / fault / task / activity` 4분류 학습이 가능한 label/window 구조를 확정하는 작업이다.

최종 결론: **{final_decision}**

## 핵심 정리

- `efd_possible`은 최상위 class label이 아니라 fault 내부 metadata로만 사용한다.
- `faults.csv`에 없는 fault disturbance는 `efd_possible=False`가 아니라 `efd_possible_unknown`으로 기록했다.
- M1 `disturbances.csv`에는 task label이 실제로 존재한다.
- normal 35건은 회사 제공 label을 그대로 유지했다.
- Event 20/34/69/67은 삭제하지 않고 fault metadata flag로 audit에 남겼다.

## Class 후보와 Window 요약

{markdown_table(source_table, ["class", "source_rows", "baseline_included", "substations", "overlap_rows"])}

## Fault 내부 Metadata

{markdown_table(fault_table, ["item", "count"])}

## Window Policy Audit

{markdown_table(window_policy_table, ["final_class", "window_policy", "rows", "coverage_eligible", "overlap_rows", "median_coverage"])}

## 최소 Baseline Sanity Check

{markdown_table(metric_table, ["feature_set", "model", "balanced_accuracy", "macro_f1", "accuracy"])}

## 산출물

| 항목 | 파일 |
| --- | --- |
| taxonomy audit | `m1_event_taxonomy_audit.csv` |
| label candidate index | `m1_4class_label_candidate_index.csv` |
| window policy audit | `m1_4class_window_policy_audit.csv` |
| feasibility summary | `m1_4class_dataset_feasibility_summary.csv` |
| baseline metrics | `m1_4class_baseline_metrics.csv` |
| baseline predictions | `m1_4class_baseline_predictions.csv` |

## 검증

- normal 35건을 유지했다.
- disturbance 165건을 audit했다.
- fault/task/activity source count를 원본과 대조했다.
- fault 67건 중 faults.csv 매칭 33건, 미매칭 34건을 확인했다.
- 미매칭 fault는 `efd_possible_unknown`으로 기록했다.
- task/activity에는 `efd_possible` label을 부여하지 않았다.
- Event 20/34/69/67이 audit에 남아 있다.
- Event 19/68/35/48 hard normal metadata를 유지했다.
- group CV에서 train/test `substation_id` overlap은 0이다.

## 한계와 다음 단계

- baseline은 label/window 설계 sanity check이며 최종 모델 선택이 아니다.
- selected window에 다른 disturbance가 겹치는 row가 있으므로, 다음 단계에서는 overlap 제거 버전과 유지 버전을 비교해야 한다.
- compact13이 충분하지 않으면 expanded154 또는 class별 feature 확장을 검토한다.
"""

REPORT_PATH.write_text(report, encoding="utf-8")
print(final_decision)


M1_4class_possible


In [9]:
required_paths = [
    EVENT_TAXONOMY_AUDIT_PATH,
    LABEL_INDEX_PATH,
    WINDOW_AUDIT_PATH,
    FEASIBILITY_SUMMARY_PATH,
    BASELINE_METRICS_PATH,
    BASELINE_PREDICTIONS_PATH,
    REPORT_PATH,
]
for path in required_paths:
    assert path.exists() and path.stat().st_size > 0, path

written_taxonomy = pd.read_csv(EVENT_TAXONOMY_AUDIT_PATH)
written_label_index = pd.read_csv(LABEL_INDEX_PATH)
written_window = pd.read_csv(WINDOW_AUDIT_PATH)
written_metrics = pd.read_csv(BASELINE_METRICS_PATH)

assert int((written_taxonomy["final_class"] == "normal").sum()) == 35
assert int((written_taxonomy["final_class"] == "fault").sum()) == 67
assert int((written_taxonomy["final_class"] == "task").sum()) == 43
assert int((written_taxonomy["final_class"] == "activity").sum()) == 55
assert int(written_taxonomy["fault_detail_available"].sum()) == 33
assert int(written_taxonomy["efd_possible_unknown"].sum()) == 34
assert int(written_taxonomy["fault_label_known"].sum()) == 31
assert int(written_taxonomy["fault_label_unknown"].sum()) == 2
assert int(written_taxonomy["efd_possible_true"].sum()) == 29
assert int(written_taxonomy["efd_possible_false"].sum()) == 4
assert set(written_taxonomy.loc[written_taxonomy["source_event_id"].isin([19, 35, 48, 68]), "hard_normal_tag"].dropna()) == {
    "review_required_normal",
    "hard_normal_metadata",
}
for event_id in [EVENT20_ID, EVENT34_ID, EVENT69_ID, EVENT67_ID]:
    assert written_taxonomy["fault_event_id"].fillna(-1).astype(int).eq(event_id).any()

task_rows = written_taxonomy.loc[written_taxonomy["final_class"].eq("task")]
activity_rows = written_taxonomy.loc[written_taxonomy["final_class"].eq("activity")]
assert not task_rows["efd_possible_true"].any()
assert not task_rows["efd_possible_false"].any()
assert not activity_rows["efd_possible_true"].any()
assert not activity_rows["efd_possible_false"].any()

assert written_window.groupby(["final_class", "window_policy"]).size().loc[("task", "task_pre_7d")] == 43
assert written_window.groupby(["final_class", "window_policy"]).size().loc[("task", "task_post_7d")] == 43
assert written_window.groupby(["final_class", "window_policy"]).size().loc[("activity", "activity_pre_7d")] == 55
assert written_window.groupby(["final_class", "window_policy"]).size().loc[("activity", "activity_post_7d")] == 55

overall_metrics = written_metrics.loc[written_metrics["metric_scope"].eq("overall")]
assert overall_metrics["group_overlap_count"].max() == 0
assert set(overall_metrics["feature_set"]) == {"compact13", "expanded154"}
assert set(overall_metrics["model"]) == {"dummy_most_frequent", "logistic_balanced"}

forbidden_terms = ["manufacturer" + "_2", "manufacturer " + "2", "M" + "2"]
check_paths = [NOTEBOOK_PATH, *required_paths]
hits = []
for path in check_paths:
    text = Path(path).read_text(encoding="utf-8", errors="ignore")
    if any(term in text for term in forbidden_terms):
        hits.append(str(path))
assert not hits, hits

validation_summary = {
    "normal": int((written_taxonomy["final_class"] == "normal").sum()),
    "fault": int((written_taxonomy["final_class"] == "fault").sum()),
    "task": int((written_taxonomy["final_class"] == "task").sum()),
    "activity": int((written_taxonomy["final_class"] == "activity").sum()),
    "fault_detail_available": int(written_taxonomy["fault_detail_available"].sum()),
    "efd_possible_unknown": int(written_taxonomy["efd_possible_unknown"].sum()),
    "final_decision": final_decision,
}
validation_summary


{'normal': 35,
 'fault': 67,
 'task': 43,
 'activity': 55,
 'fault_detail_available': 33,
 'efd_possible_unknown': 34,
 'final_decision': 'M1_4class_possible'}